In [1]:
import pandas as pd
import numpy as np

### Calculate Adjustment Factor (F) with Ideal Weights

In [2]:
def fetch_best_weights(weights_file):
    weights_data = pd.read_csv(weights_file)
    best_row = weights_data.loc[weights_data['Total Score'].idxmax()]
    weights = {
        'w1': float(best_row['Weights'].split(",")[0].split("=")[1]),
        'w2': float(best_row['Weights'].split(",")[1].split("=")[1]),
        'w3': float(best_row['Weights'].split(",")[2].split("=")[1]),
        'w4': float(best_row['Weights'].split(",")[3].split("=")[1])
    }
    return weights

def process_data(data_file, weights):
    data = pd.read_csv(data_file)

    w1, w2, w3, w4 = weights['w1'], weights['w2'], weights['w3'], weights['w4']

    # Calculate the Adjustment Factor (F)
    data['F'] = round(
        (1 + (1 / (w1 * data['OP_Price_Scaled']))) *
        (1 + w2 * data['AVPI']) *
        (1 + w3 * data['PR']) *
        (1 + w4 * data['LAR_Scaled']), 3
    )

    output_path = "../../../data/votable_supply-dhruvi/data/adjustment_factor_results.csv"
    data.to_csv(output_path, index=False)
    print(f"Results saved successfully to {output_path}")

# Main function
def main():
    weights_file = "../../../data/votable_supply-dhruvi/data/ideal_weights_results.csv"  
    data_file = "../../../data/votable_supply-dhruvi/data/indexes.csv"  

    best_weights = fetch_best_weights(weights_file)
    print(f"Best Weights: {best_weights}")

    process_data(data_file, best_weights)

if __name__ == "__main__":
    main()


Best Weights: {'w1': 0.35, 'w2': 0.3, 'w3': 0.27, 'w4': 0.08}
Results saved successfully to ../data/adjustment_factor_results.csv


### Find the Optimal Alpha Value for Ideal Weights

In [3]:
def calculate_optimal_alpha(videal_file, output_file):

    videal = pd.read_csv(videal_file)

    videal["Voriginal"] = videal["Votable Supply"]

    alpha_values = np.arange(0, 1.01, 0.01).round(2)

    results = []

    for alpha in alpha_values:

        videal["Videal"] = videal["Voriginal"] * (1 + alpha * (videal["F"] - 1))
        videal["Lower Bound"] = 1.5 * videal["Voriginal"]
        videal["Upper Bound"] = 2.0 * videal["Voriginal"]

        videal["Within Range"] = (
            (videal["Videal"] >= videal["Lower Bound"]) &
            (videal["Videal"] <= videal["Upper Bound"])
        )

        days_within_range = videal["Within Range"].sum()
        total_days = len(videal)
        percentage_within_range = (days_within_range / total_days) * 100

        results.append({
            "Alpha": alpha,
            "Days Within Range": days_within_range,
            "Percentage Within Range": percentage_within_range
        })

    results_df = pd.DataFrame(results)

    results_df.to_csv(output_file, index=False)
    print(f"Results saved to '{output_file}'")

# Main function
if __name__ == "__main__":
    videal_file = "../../../data/votable_supply-dhruvi/data/adjustment_factor_results.csv"  
    output_file = "../../../data/votable_supply-dhruvi/data/optimal_alpha_ideal_weights.csv"  
    
    calculate_optimal_alpha(videal_file, output_file)


Results saved to '../data/optimal_alpha_ideal_weights.csv'


In [4]:
def find_max_alpha(alpha_file):
    
    alpha_data = pd.read_csv(alpha_file)

    max_row = alpha_data.loc[alpha_data["Percentage Within Range"].idxmax()]

    max_percentage = max_row["Percentage Within Range"]
    corresponding_alpha = max_row["Alpha"]

    print(f"Maximum Percentage: {max_percentage:.2f}%")
    print(f"Corresponding Alpha Value: {corresponding_alpha}")

# Main function
if __name__ == "__main__":
    alpha_file = "../../../data/votable_supply-dhruvi/data/optimal_alpha_ideal_weights.csv"  # Input file path
    
    find_max_alpha(alpha_file)


Maximum Percentage: 44.35%
Corresponding Alpha Value: 0.16
